<a href="https://colab.research.google.com/github/MohamedAamer01/otto-recommender-sys/blob/main/notebooks/05b_submission_b3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Chargement des tableaux de test**

In [1]:
import os, json, time, zipfile, shutil, gc
import numpy as np, pandas as pd
from google.colab import drive, userdata
drive.mount("/content/drive")

NPY_DIR = "/content/drive/MyDrive/OTTO_Recommender_System/data/processed"
N_TEST_SESS, N_TEST = 1_671_803, 6_928_123

t_aid  = np.load(f"{NPY_DIR}/test_aid.npy")
t_size = np.load(f"{NPY_DIR}/test_sizes.npy")
assert len(t_aid) == N_TEST and len(t_size) == N_TEST_SESS and t_size.sum() == N_TEST
print("Test chargé :", f"{N_TEST_SESS:,} sessions")

Mounted at /content/drive
Test chargé : 1,671,803 sessions


**récupérer test.jsonl, extraire les identifiants, vérifier l'alignement**

In [2]:
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
DATA_DIR = "/content/otto_data"
os.makedirs(DATA_DIR, exist_ok=True)

!pip install -q --upgrade kaggle
!kaggle competitions download -c otto-recommender-system -f test.jsonl -p /content/otto_data
!kaggle competitions download -c otto-recommender-system -f sample_submission.csv -p /content/otto_data
print(os.listdir(DATA_DIR))

# Selon la version de l'API, les fichiers arrivent parfois zippés
for name in ["test.jsonl", "sample_submission.csv"]:
    zp = f"{DATA_DIR}/{name}.zip"
    if os.path.exists(zp):
        with zipfile.ZipFile(zp) as z:
            z.extractall(DATA_DIR)

test_path = f"{DATA_DIR}/test.jsonl"
assert os.path.exists(test_path), "test.jsonl introuvable"

100% 64.4M/64.4M [00:00<00:00, 86.2MB/s]

100% 12.6M/12.6M [00:00<00:00, 116MB/s]

['test.jsonl.zip', 'sample_submission.csv.zip']


**popularités par type (tout le train, et avant T pour le contrôle)**

In [3]:
import pickle
aid_arr  = np.load(f"{NPY_DIR}/aid.npy")
ts_arr   = np.load(f"{NPY_DIR}/ts.npy")
type_arr = np.load(f"{NPY_DIR}/type.npy")
sizes_arr = np.load(f"{NPY_DIR}/sizes.npy")
cum = np.concatenate([[0], np.cumsum(sizes_arr, dtype=np.int64)])

WEIGHTS = {"clicks": 0.10, "carts": 0.30, "orders": 0.60}
TYPE2ID = {"clicks": 0, "carts": 1, "orders": 2}
n_items = int(aid_arr.max()) + 1

def top_lists(mask=None, k=40):
    m = np.ones(len(aid_arr), bool) if mask is None else mask
    return {t: np.argsort(-np.bincount(aid_arr[m & (type_arr == i)], minlength=n_items),
                          kind="stable")[:k].tolist() for t, i in TYPE2ID.items()}

with open(f"{NPY_DIR}/dev_validation.pkl", "rb") as f:
    dev = pickle.load(f)
T, GT, prefix_len = dev["T"], dev["GT"], dev["prefix_len"]

tops_T   = top_lists(ts_arr < T)     # pour le contrôle sur le dev
tops_all = top_lists()               # pour la soumission : tout le train
print("Top-5 orders (tout le train) :", tops_all["orders"][:5])

Top-5 orders (tout le train) : [231487, 166037, 1733943, 1445562, 1022566]


**la fonction de prédiction et un contrôle de non-régression**

In [4]:
def b3_lists(recent_first, tops, n=20):
    """Articles de la session (récents d'abord, sans doublon), puis popularité par type."""
    seen = list(dict.fromkeys(map(int, recent_first)))[:n]
    sset = set(seen)
    out = {}
    for t in WEIGHTS:
        lst = list(seen)
        for x in tops[t]:
            if len(lst) == n: break
            if x not in sset: lst.append(x)
        out[t] = lst
    return out

# Contrôle : cette fonction doit redonner le score de B3 de la Phase 5 (0.4706)
H = {t: 0 for t in WEIGHTS}; D = {t: 0 for t in WEIGHTS}
for sid, k in prefix_len.items():
    a = cum[sid]
    lists = b3_lists(aid_arr[a:a + k][::-1], tops_T)
    for t in WEIGHTS:
        gt = GT[sid][t]
        if gt:
            H[t] += len(set(lists[t]) & gt); D[t] += min(20, len(gt))
score = sum(WEIGHTS[t] * H[t] / D[t] for t in WEIGHTS)
print(f"Score dev recalculé : {score:.4f}  (attendu 0.4706)")
assert abs(score - 0.4706) < 5e-4, "la fonction diffère de celle de la Phase 5 : ne pas soumettre"

Score dev recalculé : 0.4706  (attendu 0.4706)


In [9]:
ids = np.empty(N_TEST_SESS, dtype=np.int64)
pos = 0
with open(test_path) as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        n = len(rec["events"])
        ids[i] = rec["session"]
        assert n == t_size[i], f"taille différente à la ligne {i}"
        assert rec["events"][0]["aid"] == t_aid[pos], f"premier aid différent à la ligne {i}"
        pos += n
assert i + 1 == N_TEST_SESS and pos == N_TEST
print("Alignement OK | premiers ids :", ids[:3], "| derniers :", ids[-3:])
print("Identifiants consécutifs :", bool((np.diff(ids) == 1).all()))
np.save(f"{NPY_DIR}/test_session_ids.npy", ids)

# L'ordre des identifiants doit correspondre à celui de sample_submission
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
first = sample["session_type"].iloc[::3].str.split("_").str[0].astype("int64").values
assert np.array_equal(first, ids), "ordre différent de sample_submission"
print("Ordre identique à sample_submission, lignes :", len(sample))

Alignement OK | premiers ids : [12899779 12899780 12899781] | derniers : [14571579 14571580 14571581]
Identifiants consécutifs : True
Ordre identique à sample_submission, lignes : 5015409


**génération du fichier (écriture en flux)**

In [10]:
off = np.concatenate([[0], np.cumsum(t_size, dtype=np.int64)])
sub_path = "/content/submission_B3.csv"

t0 = time.time()
with open(sub_path, "w") as f:
    f.write("session_type,labels\n")
    for i in range(N_TEST_SESS):
        sid = ids[i]
        r = b3_lists(t_aid[off[i]:off[i + 1]][::-1], tops_all)   # toute la session visible
        for t in WEIGHTS:                                          # clicks, carts, orders
            f.write(f"{sid}_{t},{' '.join(map(str, r[t]))}\n")
        if (i + 1) % 500_000 == 0:
            print(f"{i+1:,} sessions ({time.time()-t0:.0f}s)")
print(f"Terminé en {(time.time()-t0)/60:.1f} min | taille : {os.path.getsize(sub_path)/1e6:.0f} MB")

500,000 sessions (19s)
1,000,000 sessions (34s)
1,500,000 sessions (48s)
Terminé en 0.9 min | taille : 812 MB


**vérification du format**

In [11]:
back = pd.read_csv(sub_path, usecols=["session_type"])
assert back["session_type"].equals(sample["session_type"]), "ordre ou clés différents"
print("Lignes :", back.shape[0], "(attendu 5 015 409)")

chk = pd.read_csv(sub_path, nrows=300_000)
ok = chk["labels"].map(lambda s: len(s.split()) == 20 and len(set(s.split())) == 20).all()
print("20 identifiants uniques par ligne (échantillon de 300 000) :", ok)
display(chk.head(6))

os.makedirs("/content/drive/MyDrive/OTTO_Recommender_System/submissions", exist_ok=True)
shutil.copy(sub_path, "/content/drive/MyDrive/OTTO_Recommender_System/submissions/submission_B3.csv")

Lignes : 5015409 (attendu 5 015 409)
20 identifiants uniques par ligne (échantillon de 300 000) : True


,session_type,labels
0,12899779_clicks,59625 1460571 108125 29735 485256 1733943 1849...
1,12899779_carts,59625 485256 152547 33343 166037 1733943 23148...
2,12899779_orders,59625 231487 166037 1733943 1445562 1022566 80...
3,12899780_clicks,1142000 736515 973453 582732 1460571 108125 29...
4,12899780_carts,1142000 736515 973453 582732 485256 152547 333...
5,12899780_orders,1142000 736515 973453 582732 231487 166037 173...


'/content/drive/MyDrive/OTTO_Recommender_System/submissions/submission_B3.csv'

# **Soumission**

In [15]:
!kaggle competitions submit -c otto-recommender-system -f /content/submission_B3.csv -m "B3: session items (recent first) + popularity per type"

100% 775M/775M [00:08<00:00, 92.5MB/s]
Submission ref: 58723135
98 submissions remaining today.
Successfully submitted to OTTO – Multi-Objective Recommender System

In [16]:
!kaggle competitions submissions -c otto-recommender-system

     ref  fileName           date                        description                                             status                     publicScore  privateScore  
--------  -----------------  --------------------------  ------------------------------------------------------  -------------------------  -----------  ------------  
58723135  submission_B3.csv  2026-11-22 13:27:05.023000  B3: session items (recent first) + popularity per type  SubmissionStatus.COMPLETE  0.47911      0.48035       
58722530  submission_B3.csv  2026-11-22 13:11:06.330000  B3: session items (recent first) + popularity per type  SubmissionStatus.COMPLETE  0.47911      0.48035       
